# Agent Loop Function Calling - MLflow Autologging

Ez a notebook a Databricks Foundation Model API agent loop **egyszerűsített verziója** MLflow **autologging** használatával.

## Főbb különbségek a manuális logging verziótól:

1. **Egyszerűsített kód**: Nincs manuális `mlflow.log_param()` és `mlflow.log_metric()` hívás minden tool callnál
2. **Autologging**: `mlflow.autolog()` automatikusan rögzíti a releváns információkat
3. **Kevesebb kód**: Tool függvények és API hívások egyszerűbbek, nincs benne logging logika

## Mit vár az autologging?

* Automatikus model tracking
* Automatikus paraméter és metrika rögzítés
* Egyszerűbb kód, kevesebb boilerplate

## Eredeti teljes logging verzió:
`/Users/kadarferi@gmail.com/langchain-course/N2_agent_loop_databricks_function_calling`

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage
import mlflow
import json
import time
import requests

MAX_ITERATIONS = 10
MODEL = "databricks-qwen3-next-80b-a3b-instruct"

# Initialize Databricks Workspace Client
w = WorkspaceClient()

# Autologging - csak a releváns librarykat engedélyezzük
mlflow.autolog()

In [0]:
# --- Tools (egyszerűsített - nincs manuális logging) ---

def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product, 0)


def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price and return the final price.
    Available tiers: bronze, silver, gold."""
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier, 0)
    return round(price * (1 - discount / 100), 2)

In [0]:
# OpenAI-kompatibilis JSON schema

tools_for_llm = [
    {
        "type": "function",
        "function": {
            "name": "get_product_price",
            "description": "Look up the price of a product in the catalog.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product": {
                        "type": "string",
                        "description": "The product name, e.g. 'laptop', 'headphones', 'keyboard'",
                    },
                },
                "required": ["product"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_discount",
            "description": "Apply a discount tier to a price and return the final price. Available tiers: bronze, silver, gold.",
            "parameters": {
                "type": "object",
                "properties": {
                    "price": {"type": "number", "description": "The original price"},
                    "discount_tier": {
                        "type": "string",
                        "description": "The discount tier: 'bronze', 'silver', or 'gold'",
                    },
                },
                "required": ["price", "discount_tier"],
            },
        },
    },
]

In [0]:
# Databricks Foundation Model API hívás (egyszerűsített)

def databricks_chat(messages):
    """Call Databricks Foundation Model API with tool support."""
    workspace_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)
    
    url = f"{workspace_url}/serving-endpoints/{MODEL}/invocations"
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    body = {
        "messages": messages,
        "max_tokens": 10000,
        "tools": tools_for_llm
    }
    
    response = requests.post(url, headers=headers, json=body)
    response.raise_for_status()
    
    return response.json()

In [0]:
def run_agent(question: str):
    """Agent loop - csak autologging, semmi manuális MLflow hívás."""
    
    tools_dict = {
        "get_product_price": get_product_price,
        "apply_discount": apply_discount,
    }

    print(f"Question: {question}")
    print("=" * 60)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful shopping assistant. "
                "You have access to a product catalog tool "
                "and a discount tool.\n\n"
                "STRICT RULES — you must follow these exactly:\n"
                "1. NEVER guess or assume any product price. "
                "You MUST call get_product_price first to get the real price.\n"
                "2. Only call apply_discount AFTER you have received "
                "a price from get_product_price. Pass the exact price "
                "returned by get_product_price — do NOT pass a made-up number.\n"
                "3. NEVER calculate discounts yourself using math. "
                "Always use the apply_discount tool.\n"
                "4. If the user does not specify a discount tier, "
                "ask them which tier to use — do NOT assume one."
            ),
        },
        {"role": "user", "content": question},
    ]

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n--- Iteration {iteration} ---")

        # API hívás (autolog automatikusan követi?)
        response = databricks_chat(messages=messages)
        
        # Parse response
        choices = response.get("choices", [])
        if not choices:
            print("ERROR: No response choices returned")
            return None
            
        ai_message = choices[0].get("message", {})
        tool_calls = ai_message.get("tool_calls", [])

        # Végső válasz
        if not tool_calls:
            final_answer = ai_message.get("content", "")
            print(f"\nFinal Answer: {final_answer}")
            return final_answer

        # Tool call feldolgozás
        tool_call = tool_calls[0]
        tool_name = tool_call.get("function", {}).get("name", "")
        tool_args_str = tool_call.get("function", {}).get("arguments", "{}")
        tool_call_id = tool_call.get("id", "")
        
        tool_args = json.loads(tool_args_str) if isinstance(tool_args_str, str) else tool_args_str

        print(f"  [Tool Selected] {tool_name} with args: {tool_args}")

        tool_to_use = tools_dict.get(tool_name)
        if tool_to_use is None:
            raise ValueError(f"Tool '{tool_name}' not found")

        # Tool végrehajtás (autolog automatikusan követi?)
        observation = tool_to_use(**tool_args)
        print(f"  [Tool Result] {observation}")

        # Üzenetek frissítése
        messages.append({
            "role": "assistant",
            "content": ai_message.get("content", "") or "",
            "tool_calls": [{"id": tool_call_id, "type": "function", "function": {"name": tool_name, "arguments": tool_args_str}}]
        })
        messages.append({
            "role": "tool",
            "content": str(observation),
            "tool_call_id": tool_call_id
        })

    print("ERROR: Max iterations reached")
    return None

In [0]:
# Teszt futtatás
print("Hello Databricks Agent (Autologging)!")
print()
result = run_agent("What is the price of a laptop after applying a gold discount?")